<a href="https://colab.research.google.com/github/bei931016/MachineLearning/blob/main/0702_Colab_LINE_Bot_with_GEMINI_Tooluse%20copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [9]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [10]:
import os
from pyngrok import ngrok

In [11]:
ngrok.kill()

In [12]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://extinct-perkiness-shrapnel.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://extinct-perkiness-shrapnel.ngrok-free.dev


True

In [13]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [14]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [15]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學 (Ming Hsin University of Science and Technology, 簡稱明新科大) 是一所位於台灣新竹縣的私立科技大學。

以下是其簡要介紹：

1.  **創校歷史與發展：**
    *   **創立時間：** 1966年，以「明新工業專科學校」之名成立。
    *   **發展歷程：** 歷經多年發展與轉型，於1997年改制為「明新技術學院」，並於2002年正式升格為「明新科技大學」。其發展軌跡反映了台灣技職教育從專科到科技大學的演變。

2.  **地理位置：**
    *   座落於新竹縣新豐鄉，地理位置鄰近新竹科學園區、湖口工業區等重要產業聚落，這使得學校在產學合作及學生就業上具有獨特優勢。

3.  **教學與學術特色：**
    *   **辦學理念：** 明新科大秉持「勤學、樂觀、力行」的校訓，強調理論與實務並重，致力於培養具備專業技能、人文素養及國際觀的科技人才。
    *   **學院與學系：** 目前設有工程學院、管理學院、服務事業學院及設計學院等四大教學單位，涵蓋了電機電子、機械、資訊、化學工程、工商管理、觀光餐旅、幼兒保育、多媒體設計等多樣化的學術領域。
    *   **產學合作：** 因地利之便，學校與周邊的科技產業、服務業等建立了緊密的產學合作關係，提供學生豐富的實習機會，並將業界最新技術融入教學，提升學生職場競爭力。
    *   **實務導向：** 強調實作能力培養，設有完善的實驗室及實習工廠，鼓勵學生動手實踐，學以致用。
    *   **國際交流：** 積極推動國際學術交流與合作，提供學生海外研修、交換等機會，拓寬國際視野。

4.  **辦學績效：**
    *   明新科大在私立技職院校中擁有良好的口碑，畢業生在業界表現獲得肯定，是桃竹苗地區重要的技職人才培育基地。學校也積極參與各項評鑑，力求提升教學品質與學術研究水準。

總結來說，明新科技大學是一所歷史悠久、務實辦學，並與產業脈動緊密結合的科技大學，致力於為國家及區域發展培養具備專業素養與實務能力的科技與服務人才。


In [16]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學目前的校長是 **劉國偉** 博士。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:22:46] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:23:08] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617021180691873862","quoteToken":"SFkbv-TUrMp0Udi5ysIt9MXFXX9e03morc48FRgfiRMmapa-EhjYlv161lsF7CKI5PauVUN7xjh0Jidmy3coGDBCFyhv0F40B-Bo3deGN0bWre64jZfa5GlRkDnoEv17dKg9BUN3vYysFW615Fd4_A","markAsReadToken":"urTM0zryua4tZOAeL9TI3YQ9lFdJZYf20b6StGLN0jAPWfHdroJQjIfKE4YKe3M3t-Fo2LjFWUYavebFUPpo5aV_wWICkex2lQhY_WBkEdvP35y9WNuUwf1DggX0Wv2e80aj05N9zK7hTsi8XPp7gvDbFUo43ad9R4RvwepdktrshgDYRYdvb4v2i-W-3YIbpSASSRHtcKxnYpLFRzgDcw","text":"簡介明新科技大學"},"webhookEventId":"01KTA4VKE02NP492G9QF64708H","deliveryContext":{"isRedelivery":false},"timestamp":1780604586949,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"4f3baf767add4656bbac07a54ba258cb","mode":"active"}]}


...

INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:24:32] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[]}
BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617021360057090100","quoteToken":"9AOskpRYTWJuP0BBPNwb84fZ7XGhjeIiTIgpDs2q28lOCsCxi0pNdY2_Uqvz1x6M3WT-iKdyrLqqgRs7R5ngwfWKjdDP8-2KGFDdMVgUi6ahx2Mp6e-ikfQGuOFgYRQi1KYLqSLUfgofEwOu8gyblA","markAsReadToken":"CLmJ7woAuMzj85BX2wE9DFu5GBKVUvLHHUirVRthFlcm6yJx75JH5ofW0QAEjNUG--FLWkI48vwDV13Ehpwaa8YSvJzW1Nw7NUc5ZH-JKZAyI8zd1qQNjquopuYbLFO7sPJEy2T8j-sbdkl0Hx--FQVngv0-tYU4pI1Gs9o46f3RSR1UOWcw8Gx77BqGJCjQcyNivKKIoGrNzRU3Nr9oJQ","text":"AI 簡介明新科技大學20字以內"},"webhookEventId":"01KTA4YVTZ30YXD860GJZ30PVQ","deliveryContext":{"isRedelivery":false},"timestamp":1780604693859,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"baf5e5a60dbf49f497a058fe62bfd7e7","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:24:59] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617021415673561123","quoteToken":"kLyDFmglxFxkWip6Iikgev48D1Vk_hBjXFqI4DnbOcDLvmmlBf7JKi2_TwiFDx6SVcuxQcKRsKWd_93f2PKl06Y_tVu_CEBnLV5N5objHJ8x0VGrEFvTDO_-J_Th-ZgCk_U3jjF6RJw3jrsn2S0ICw","markAsReadToken":"fhrJAvY1OjhrWMQzW4EEqo0l1jaBhPEhThJh3RqsocujS1Zy_g9Al4tDB7Zzr5fSirYziwU0iVX_mhwXZkTrwHgSA2SC-C007WUziK_asTUD6DagLNa2vdeQRVEq-zvrzxVDrTs_T8obz7-moKNbkNV7mZMZtFtv_h1n9Q3ERz553pnhFeoLTRkxu3Zk0Jffddjs03lp3nC0_QAHrgnrAA","text":"AI 校長是誰"},"webhookEventId":"01KTA4ZW6X4HWHWF10B5G26VGM","deliveryContext":{"isRedelivery":false},"timestamp":1780604727007,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"6b097cfd8ed7446594396ba26ce56205","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:25:29] "POST / HTTP/1.1" 200 -
